# Model Explainability and Error Analysis

This notebook analyzes the saved MNIST CNN model without retraining it.

## 1. Introduction

The goal is to review errors, confidence behavior, per-class performance, and lightweight saliency maps for the selected baseline CNN.

## 2. Load Saved Model and Test Data

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import json
import matplotlib.pyplot as plt
import pandas as pd

from src.analysis.confidence_analysis import run_confidence_analysis
from src.analysis.error_analysis import generate_predictions, load_model_and_test_data, run_error_analysis
from src.analysis.model_report import generate_model_reports
from src.utils.paths import IMAGES_DIR, REPORTS_DIR

model, x_test, y_test, x_raw = load_model_and_test_data()
x_test.shape, y_test.shape, x_raw.shape

## 3. Generate Predictions

In [ ]:
y_pred, probabilities, confidence_scores = generate_predictions(model, x_test)
pd.Series(y_pred).value_counts().sort_index()

## 4. Overall Performance Review

In [ ]:
training_summary = json.loads((REPORTS_DIR / "training_summary.json").read_text())
evaluation_summary = json.loads((REPORTS_DIR / "evaluation_summary.json").read_text())

pd.DataFrame([
    {
        "training_accuracy": training_summary["final_training_accuracy"],
        "validation_accuracy": training_summary["final_validation_accuracy"],
        "test_accuracy": evaluation_summary["test_accuracy"],
        "test_loss": evaluation_summary["test_loss"],
    }
])

## 5. Per-Class Performance

In [ ]:
confidence_summary = run_confidence_analysis()
per_class_df = pd.read_csv(REPORTS_DIR / "explainability" / "per_class_performance.csv")
per_class_df

## 6. Confidence Analysis

In [ ]:
confidence_summary

## 7. Misclassified Samples

In [ ]:
error_summary = run_error_analysis()
misclassified_df = pd.read_csv(REPORTS_DIR / "error_analysis" / "misclassified_samples.csv")
misclassified_df.head(10)

## 8. Common Confusion Pairs

In [ ]:
confusion_pairs = pd.read_csv(REPORTS_DIR / "error_analysis" / "top_confusion_pairs.csv")
confusion_pairs.head(10)

## 9. Saliency Map Examples

In [ ]:
report_summary = generate_model_reports()
saliency_path = IMAGES_DIR / "explainability" / "saliency_examples.png"

img = plt.imread(saliency_path)
plt.figure(figsize=(8, 8))
plt.imshow(img)
plt.axis("off")
plt.show()

## 10. Key Findings

In [ ]:
report_summary

## 11. Final Model Decision

`mnist_cnn_baseline` remains the selected final model based on strong test accuracy, low error rate, and deployment-friendly model size.

## 12. Limitations and Next Steps

- MNIST is cleaner than real-world handwriting.
- The model supports digits only.
- Saliency maps show pixel sensitivity, not causal explanation.
- Future work should include EMNIST, augmentation, calibration, robustness checks, and deployment monitoring.